# Date aligment

### Setup

In [99]:
import os
import datetime
from datetime import time
import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv
import re
from time import sleep

In [100]:
# load sec user (name + email)
load_dotenv()
SEC_USER_AGENT = os.environ["SEC_USER_AGENT"]

# time zone of edgar
NY = "America/New_York"

## EDA

In [101]:
# getting the list of the us dates with the market open from a sp500 fund
sp500 = yf.download("SPY", start="2008-01-01", end="2026-01-01")

# normalize for precaution (date to timestamp 00:00:00)
sessions = sp500.index.normalize()
sessions[0]

[*********************100%***********************]  1 of 1 completed


Timestamp('2008-01-02 00:00:00')

In [122]:
# Positive controls: regular sessions must be in the calendar
assert pd.Timestamp("2019-07-03") in sessions   # day before the 4th of July
assert pd.Timestamp("2018-12-24") in sessions   # Christmas Eve (half day, but open)

# Holidays and special closures must be missing
assert pd.Timestamp("2019-07-04") not in sessions   # 4th of July
assert pd.Timestamp("2018-12-05") not in sessions   # George H. W. Bush funeral
assert pd.Timestamp("2018-12-25") not in sessions   # Christmas
assert pd.Timestamp("2019-04-19") not in sessions   # Good Friday

In [108]:
def edgar_filings(cik, start="2008-01-01", end="2026-01-01"):
    headers = {"User-Agent": SEC_USER_AGENT}                     # SEC identification
    url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json" # company index padded to 10 digits
    data = requests.get(url, headers=headers).json()             # dict: company info + filings

    # recent: the newest ~1000 filings
    pages = [pd.DataFrame(data["filings"]["recent"])]

    # files: older pages
    for page in data["filings"]["files"]:
        # keep only the pages whose date range overlaps [start, end] 
        if page["filingTo"] >= start and page["filingFrom"] <= end:
            url = f"https://data.sec.gov/submissions/{page['name']}"            
            pages.append(pd.DataFrame(requests.get(url, headers=headers).json()))  # same columns as "recent"
            sleep(0.15)   # SEC limit: 10 requests per second

    # recent + old pages in one DataFrame (gap between them is possible, e.g. JPM 2020-2025)
    filings = pd.concat(pages, ignore_index=True)
    
    # Cut rows with newer dates than end and older than start
    filings =  filings[filings["filingDate"]>= start]
    filings =  filings[filings["filingDate"]<= end]

    return filings

In [109]:
companies = {"Apple": 320193, "JP Morgan": 19617, "Duke Energy": 1326160}

filings_all = pd.concat(
    [edgar_filings(company_cik).assign(company=name) for name, company_cik in companies.items()],
    ignore_index=True,
)
filings_all.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription,company
0,0001140361-25-044561,2025-12-05,2025-12-04,2025-12-05T21:31:42.000Z,34,8-K,001-36743,251553428,5.02,XBRL,238615,1,1,NaN,ef20060722_8k.htm,8-K,Apple
1,0001462356-25-000012,2025-11-14,2025-11-12,2025-11-14T23:30:15.000Z,,4,,,,4,4618,0,0,NaN,xslF345X05/wk-form4_1763163012.xml,FORM 4,Apple
2,0001354457-25-001138,2025-11-14,,2025-11-14T21:10:12.000Z,34,25-NSE,000-10030,251486148,,25-NSE,3106,0,0,NaN,xslF25X02/primary_doc.xml,,Apple
3,0001631982-25-000011,2025-11-12,2025-11-07,2025-11-12T23:30:10.000Z,,4,,,,4,5135,0,0,NaN,xslF345X05/wk-form4_1762990206.xml,FORM 4,Apple
4,0000320193-25-000079,2025-10-31,2025-09-27,2025-10-31T10:01:26.000Z,34,10-K,001-36743,251437791,,XBRL,9392337,1,1,NaN,aapl-20250927.htm,10-K,Apple


### Dictionary
1. **accessionNumber**: unique ID of the filing in EDGAR (filer agent CIK - year - sequence).
2. **filingDate**: date the filing was officially recorded by the SEC.
3. **reportDate**: end of the period the filing covers (e.g. fiscal year end for a 10-K); empty for some forms.
4. **acceptanceDateTime**: exact timestamp (UTC) when EDGAR accepted the submission; key for aligning with trading sessions.
5. **act**: securities law the filing is made under (e.g. 33 = Securities Act 1933, 34 = Exchange Act 1934).
6. **form**: form type (10-K, 10-Q, 8-K, 4, 144...).
7. **fileNumber**: SEC file number of the registrant/registration.
8. **filmNumber**: internal SEC microfilm/document number.
9. **items**: items reported (mainly for 8-K, e.g. "2.02,9.01").
10. **core_type**: base form type, without amendment suffix.
11. **size**: size of the submission in bytes.
12. **isXBRL**: 1 if the filing includes XBRL financial data.
13. **isInlineXBRL**: 1 if the XBRL is embedded in the HTML document (iXBRL).
14. **isXBRLNumeric**: 1 if the XBRL contains numeric financial facts.
15. **primaryDocument**: file name of the main document of the filing.
16. **primaryDocDescription**: short description of the main document (e.g. "10-K", "FORM 4").
17. **company**: company name (added in the concat)

In [110]:
ten_k = filings_all[filings_all["form"]=="10-K"]
ten_k.info()

<class 'pandas.DataFrame'>
Index: 54 entries, 4 to 146588
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   accessionNumber        54 non-null     str   
 1   filingDate             54 non-null     str   
 2   reportDate             54 non-null     str   
 3   acceptanceDateTime     54 non-null     str   
 4   act                    54 non-null     str   
 5   form                   54 non-null     str   
 6   fileNumber             54 non-null     str   
 7   filmNumber             54 non-null     str   
 8   items                  54 non-null     str   
 9   core_type              54 non-null     str   
 10  size                   54 non-null     int64 
 11  isXBRL                 54 non-null     int64 
 12  isInlineXBRL           54 non-null     int64 
 13  isXBRLNumeric          0 non-null      object
 14  primaryDocument        54 non-null     str   
 15  primaryDocDescription  54 non-null   

**Findings**   
 - Dates are imported as strings  
 - acceptanceDateTime is written in timestamp format with the "Z" at the end, suggesting UCT timezone

## Aligment function

In [111]:
def first_tradable_date(accepted_at: str | pd.Timestamp, sessions: pd.DatetimeIndex) -> datetime.date:
    """
    Return the first session whose 09:30 NY open is after `accepted_at`.

    Parameters
    ----------
    accepted_at : str or pandas.Timestamp
        Filing acceptance time (naive values are assumed NY time).
    sessions : pandas.DatetimeIndex
        Sorted trading session dates.

    Returns
    -------
    datetime.date
        First tradable session date.
    """

    # Adapt data types
    accepted_at = pd.Timestamp(accepted_at) # assert is a timestamp

    sessions = pd.DatetimeIndex(pd.to_datetime(sessions)) # sum timedeltas

    # Newyork timezone 
    # If a timezone is already set, corvert it first to avoid errors

    if accepted_at.tz is None:
        accepted_at = accepted_at.tz_localize(NY)      # no zone: assumes new york
    else:
        accepted_at = accepted_at.tz_convert(NY)       # with zone: convert it no NY

    # Newyork timezone + open hour (9:30)
    opens = (sessions + pd.Timedelta(hours=9, minutes=30)).tz_localize(NY)

    # first open index after accepted_at
    i = opens.searchsorted(accepted_at, side="right")

    # catch error if the date is in the future
    if i == len(opens):
        raise ValueError(f"{accepted_at} is after the last session in the calendar ({sessions[-1].date()})")

    return sessions[i].date()

In [112]:
first_tradable_dates = [first_tradable_date(date, sessions) for date in ten_k["acceptanceDateTime"]]
first_tradable_dates[:5]

[datetime.date(2025, 10, 31),
 datetime.date(2024, 11, 1),
 datetime.date(2023, 11, 3),
 datetime.date(2022, 10, 28),
 datetime.date(2021, 10, 29)]

### Test 

The table below describes 6 scenarios:

| Accepted (NY time) | Expected | Why |
| --- | --- | --- |
| 2019-07-01 08:00 | 2019-07-01 | Before the open: same day |
| 2019-07-01 09:30 | 2019-07-02 | Exactly at the open: the session has already started |
| 2019-07-01 11:00 | 2019-07-02 | During the session |
| 2019-07-01 16:00 | 2019-07-02 | At the close |
| 2019-07-03 17:00 | 2019-07-05 | Holiday eve: skips July 4th |
| 2019-07-05 17:00 | 2019-07-08 | Friday evening: next Monday |

In [ ]:
# Test if it works as intended
accepted_dates = pd.to_datetime([
    "2019-07-01 08:00", 
    "2019-07-01 09:30",
    "2019-07-01 11:00",
    "2019-07-01 16:00",
    "2019-07-03 17:00",
    "2019-07-05 17:00",
]).tz_localize(NY)

dates = [first_tradable_date(date, sessions) for date in accepted_dates]
dates

[datetime.date(2019, 7, 1),
 datetime.date(2019, 7, 2),
 datetime.date(2019, 7, 2),
 datetime.date(2019, 7, 2),
 datetime.date(2019, 7, 5),
 datetime.date(2019, 7, 8)]

**Findings**   
 - All dates are the expected ones, thefore it works

## True Acceptance hour of EDGAR
The `acceptanceDateTime` field of the fillings contains "z" which indicates UCT, not New York. Since their web contains it in New York's datetime.

In this section both api and web version `acceptanceDateTime` will be compared using `first_trazable_date`, there should be a 4h difference (5 in winter schedule) if it is truly a UCT. 

### API

In [114]:
# Apple will be used as an example to explore this hypothesis
cik = 320193 # Apple 
url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
data = requests.get(url, headers={"User-Agent": SEC_USER_AGENT}).json()

In [115]:
filings = pd.DataFrame(data["filings"]["recent"])
filings.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription
0,0001354457-26-001062,2026-11-09,,2026-11-09T22:00:55.000Z,34,25-NSE,000-10030,261536843,,25-NSE,3110,0,0,0.0,xslF25X02/primary_doc.xml,
1,0001140361-26-042798,2026-11-09,2026-11-05,2026-11-09T21:30:40.000Z,34,8-K,001-36743,261536464,"8.01,9.01",XBRL,1029343,1,1,0.0,ef20085077_8k.htm,
2,0001140361-26-042505,2026-11-06,,2026-11-06T21:30:47.000Z,33,424B2,333-282937,261532487,,424B2,930109,1,1,0.0,ny20084621x2_424b2.htm,424B2
3,0001140361-26-042173,2026-11-06,,2026-11-06T02:54:06.000Z,34,FWP,333-282937,261529143,,FWP,112850,0,0,0.0,ny20084621x3_fwp.htm,FWP
4,0001140361-26-042149,2026-11-05,2026-11-03,2026-11-05T23:30:13.000Z,,4,,,,4,4822,0,0,0.0,xslF345X06/form4.xml,FORM 4


In [116]:
# Filter for 10k and date to avoid dates in the future
ten_k = filings[filings["form"] == "10-K"]
ten_k = ten_k[ten_k["filingDate"] < "2026-01-01"]
ten_k.head()

,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription
94,0000320193-25-000079,2025-10-31,2025-09-27,2025-10-31T10:01:26.000Z,34,10-K,001-36743,251437791,,XBRL,9392337,1,1,NaN,aapl-20250927.htm,10-K
179,0000320193-24-000123,2024-11-01,2024-09-28,2024-11-01T10:01:36.000Z,34,10-K,001-36743,241416806,,XBRL,9759333,1,1,NaN,aapl-20240928.htm,10-K
273,0000320193-23-000106,2023-11-03,2023-09-30,2023-11-02T22:08:27.000Z,34,10-K,001-36743,231373899,,XBRL,9569569,1,1,NaN,aapl-20230930.htm,10-K
351,0000320193-22-000108,2022-10-28,2022-09-24,2022-10-27T22:01:14.000Z,34,10-K,001-36743,221338448,,XBRL,10332356,1,1,NaN,aapl-20220924.htm,10-K
426,0000320193-21-000105,2021-10-29,2021-09-25,2021-10-28T22:04:28.000Z,34,10-K,001-36743,211359752,,XBRL,10502096,1,1,NaN,aapl-20210925.htm,10-K


In [117]:
first_tradable_days_appl = [first_tradable_date(date, sessions) for date in ten_k["acceptanceDateTime"]]
first_tradable_days_appl

[datetime.date(2025, 10, 31),
 datetime.date(2024, 11, 1),
 datetime.date(2023, 11, 3),
 datetime.date(2022, 10, 28),
 datetime.date(2021, 10, 29),
 datetime.date(2020, 10, 30),
 datetime.date(2019, 10, 31),
 datetime.date(2018, 11, 5),
 datetime.date(2017, 11, 3),
 datetime.date(2016, 10, 27),
 datetime.date(2015, 10, 29)]

### Web

In [118]:
def web_acceptance(cik: int, acc: str) -> pd.Timestamp:
    """
    Return the EDGAR acceptance time of a filing as shown on its web index page.

    Parameters
    ----------
    cik : int
        Company CIK.
    acc : str
        Filing accession number (with dashes).

    Returns
    -------
    pandas.Timestamp
        Acceptance time, localized to New York.
    """
    url = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"   # company folder
           f"{acc.replace('-', '')}/{acc}-index-headers.html")            # filing folder (no dashes) + header page
    
    html = requests.get(url, headers={"User-Agent": SEC_USER_AGENT}).text # raw html of the page
    
    raw = re.search(r"ACCEPTANCE-DATETIME>\s*(\d{14})", html).group(1)    # 14 digits after the tag: Year Month Day Hour Minute Second 

    parsed = pd.to_datetime(raw, format="%Y%m%d%H%M%S")                   # parse
    return parsed.tz_localize(NY)     # Return in New York timezome

web_dates = []
for acc in ten_k["accessionNumber"]:
    web_dates.append(web_acceptance(cik, acc))
    sleep(0.15)  # SEC limit: 10 req/s

In [119]:
web_dates

[Timestamp('2025-10-31 06:01:26-0400', tz='America/New_York'),
 Timestamp('2024-11-01 06:01:36-0400', tz='America/New_York'),
 Timestamp('2023-11-02 18:08:27-0400', tz='America/New_York'),
 Timestamp('2022-10-27 18:01:14-0400', tz='America/New_York'),
 Timestamp('2021-10-28 18:04:28-0400', tz='America/New_York'),
 Timestamp('2020-10-29 18:06:25-0400', tz='America/New_York'),
 Timestamp('2019-10-30 18:12:36-0400', tz='America/New_York'),
 Timestamp('2018-11-05 08:01:40-0500', tz='America/New_York'),
 Timestamp('2017-11-03 08:01:37-0400', tz='America/New_York'),
 Timestamp('2016-10-26 16:42:16-0400', tz='America/New_York'),
 Timestamp('2015-10-28 16:31:09-0400', tz='America/New_York')]

### Comparison

In [120]:
comparison = ten_k[["accessionNumber", "acceptanceDateTime"]].assign(
    api_utc=pd.to_datetime(ten_k["acceptanceDateTime"]),
    web_ny=web_dates,
)
comparison["diff"] = comparison["api_utc"] - comparison["web_ny"]  # should be 0 if both mark the same instant
comparison

,accessionNumber,acceptanceDateTime,api_utc,web_ny,diff
94,0000320193-25-000079,2025-10-31T10:01:26.000Z,2025-10-31 10:01:26+00:00,2025-10-31 06:01:26-04:00,0 days
179,0000320193-24-000123,2024-11-01T10:01:36.000Z,2024-11-01 10:01:36+00:00,2024-11-01 06:01:36-04:00,0 days
273,0000320193-23-000106,2023-11-02T22:08:27.000Z,2023-11-02 22:08:27+00:00,2023-11-02 18:08:27-04:00,0 days
351,0000320193-22-000108,2022-10-27T22:01:14.000Z,2022-10-27 22:01:14+00:00,2022-10-27 18:01:14-04:00,0 days
426,0000320193-21-000105,2021-10-28T22:04:28.000Z,2021-10-28 22:04:28+00:00,2021-10-28 18:04:28-04:00,0 days
495,0000320193-20-000096,2020-10-29T22:06:25.000Z,2020-10-29 22:06:25+00:00,2020-10-29 18:06:25-04:00,0 days
566,0000320193-19-000119,2019-10-30T22:12:36.000Z,2019-10-30 22:12:36+00:00,2019-10-30 18:12:36-04:00,0 days
636,0000320193-18-000145,2018-11-05T13:01:40.000Z,2018-11-05 13:01:40+00:00,2018-11-05 08:01:40-05:00,0 days
724,0000320193-17-000070,2017-11-03T12:01:37.000Z,2017-11-03 12:01:37+00:00,2017-11-03 08:01:37-04:00,0 days
869,0001628280-16-020309,2016-10-26T20:42:16.000Z,2016-10-26 20:42:16+00:00,2016-10-26 16:42:16-04:00,0 days


**Findings**  
 - The EDGAR API returns a True UCT Timestamp since the difference is 0

## Reports published with the market closed
This shows why the 4-5 hour difference matters

In [121]:
acceptance_utc = pd.to_datetime(ten_k["acceptanceDateTime"])   # str -> tz-aware UTC
acceptance_ny = acceptance_utc.dt.tz_convert(NY)               # wall-clock time in New York

# market closed: before the 09:30 open or at/after the 16:00 close
t_utc = acceptance_utc.dt.time
t_ny = acceptance_ny.dt.time
closed_utc = (t_utc < time(9, 30)) | (t_utc >= time(16, 0))  # wrong: reads the UTC clock
closed_ny = (t_ny < time(9, 30)) | (t_ny >= time(16, 0))     # right: reads the NY clock

print(f"Market Closed (wrong): {closed_utc.mean() * 100 :.5} %")
print(f"Market Closed (right): {closed_ny.mean() * 100} %")

Market Closed (wrong): 63.636 %
Market Closed (right): 100.0 %


**Findings**
 - In this sample, all 10-K reports were accepted with the market closed: before the 09:30 open or after the 16:00 close.
 - Reading the UTC timestamp as if it were New York time shifts every acceptance 4-5 hours later. The four reports accepted before the open (06:01 and 08:01 NY) would appear to be accepted during the session, and would be assigned to the **next** session instead of the same day.
 - This is not look-ahead bias, but it is still a serious error: the target window would start one day late and miss the first trading day after the release, which is usually the day with the largest reaction.